# 🏐 Ball detector — generation 2
**Why:** first full-game scorecard showed the ball detector is the bottleneck
(median 3.7 detections/s; contacts P51/R57, everything downstream cascades).

**What this does:** mines a bigger training set from ALL processed games using
the current model at low confidence + physics (parabola) verification, adds
hard negatives, then fine-tunes a new model **at native 1080p** and scores it
(with a confidence sweep) against your reviewed corrections BEFORE you promote.
Mining + training run at high resolution so the model learns the ball at the
same scale it's detected — the fix for the precision drop seen at inference-only.

**Prereqs in Drive/balltime:** game videos, `bundles/`, latest `vbpipe.zip`,
`ball_model.pt`, and `corrections_*.json` (named to match each video stem,
e.g. game1.mp4 → corrections_game1.json).

**You don't have to get this perfect up front — Cell 1 runs a preflight
check and prints exactly what's present, what's missing, and where to put
it. Just set Runtime → T4 GPU, then Runtime → Run all, and read Cell 1's
output.** ~30-45 min.

In [ ]:
# Cell 1 — setup & preflight  (run this first; it tells you exactly what's missing)
from google.colab import drive
drive.mount("/content/drive")
import os, glob, zipfile, json

D = "/content/drive/MyDrive/balltime"
def has(p): return os.path.exists(p)
problems, warnings = [], []

# ── 0. the balltime folder itself ────────────────────────────────────────────
if not has(D):
    problems.append(
        f"Can't find your Drive folder at:\n      {D}\n"
        "   Fix: in Google Drive (the same account you just signed in with),\n"
        "   make a folder called 'balltime' at the top level of 'My Drive'.\n"
        "   Everything below goes inside it.")

# ── 1. the two required files that live in Drive/balltime ─────────────────────
#     vbpipe.zip     = the pipeline code (rebuilt on Ken's machine each session)
#     ball_model.pt  = the current ball detector (this notebook improves on it)
for fname, what in [("vbpipe.zip", "the pipeline code"),
                    ("ball_model.pt", "the current ball detector")]:
    if not has(f"{D}/{fname}"):
        problems.append(
            f"Missing {fname} ({what}).\n"
            f"   Fix: upload {fname} into  {D}/")

# ── 2. games = a bundle zip + its matching video ─────────────────────────────
#     bundle:  Drive/balltime/bundles/game_bundle_<stem>.zip   (has game.json)
#     video:   Drive/balltime/<stem>.mp4  (or .mov / .mkv)
#     The <stem> is the game's name (e.g. game1 -> game1.mp4).
bundles = sorted(glob.glob(f"{D}/bundles/game_bundle_*.zip"))
VID_EXT = (".mp4", ".mov", ".mkv")
bundle_stems, games = {}, {}
if not bundles:
    warnings.append(
        "No bundles found in Drive/balltime/bundles/.\n"
        "   Training mines arcs from every game's video, so add at least one:\n"
        "   put game_bundle_<name>.zip in Drive/balltime/bundles/ and the\n"
        "   matching <name>.mp4 in Drive/balltime/.")
for b in bundles:
    stem = os.path.basename(b)[len("game_bundle_"):-4]
    bundle_stems[stem] = b
    vid = next((f"{D}/{stem}{e}" for e in VID_EXT if has(f"{D}/{stem}{e}")), None)
    if not vid:
        warnings.append(
            f"Bundle for '{stem}' has no video — expected {stem}.mp4 in {D}/\n"
            f"   Upload {stem}.mp4 (or .mov/.mkv) so '{stem}' can be used.")
        continue
    try:
        with zipfile.ZipFile(b) as z: gjson = json.loads(z.read("game.json"))
    except Exception as e:
        warnings.append(f"Bundle for '{stem}' is unreadable (missing game.json?): {e}"); continue
    games[stem] = (vid, gjson)   # (video path, parsed game.json)

# ── 3. corrections = your reviewed labels (used to SCORE the new model) ───────
#     Drive/balltime/corrections_<stem>.json   (stem must match the video!)
corr_files = glob.glob(f"{D}/corrections_*.json")
corrs = {}
for f in corr_files:
    stem = os.path.basename(f)[len("corrections_"):-5]
    try:
        corrs[stem] = json.load(open(f))
    except Exception as e:
        warnings.append(f"Couldn't read {os.path.basename(f)}: {e}")

# ── report ───────────────────────────────────────────────────────────────────
print("=" * 64)
print("PREFLIGHT — what's in Drive/balltime")
print("=" * 64)

# per-game readiness table (union of everything we saw a name for)
all_stems = sorted(set(bundle_stems) | set(games) | set(corrs))
if all_stems:
    print(f"\n{'game':<16}{'bundle':<9}{'video':<9}{'corrections':<13}status")
    print("-" * 64)
    for s in all_stems:
        b_ok, v_ok, c_ok = s in bundle_stems, s in games, s in corrs
        if v_ok and c_ok:   status = "✅ trains + scored"
        elif v_ok:          status = "✅ trains (no review to score against)"
        elif c_ok and b_ok: status = "⚠️  need video " + s + ".mp4"
        elif c_ok:          status = "⚠️  need bundle + video for " + s
        else:               status = "⚠️  need video"
        chk = lambda ok: "  yes  " if ok else "  --   "
        print(f"{s:<16}{chk(b_ok):<9}{chk(v_ok):<9}{chk(c_ok):<13}{status}")
else:
    print("\n(no games, bundles, or corrections found yet)")

# corrections whose stem matches nothing = the classic 'skip: no video/bundle'
orphan_corr = [s for s in corrs if s not in games]
if orphan_corr:
    warnings.append(
        "These corrections have no matching video+bundle, so Cell 5 will\n"
        f"   SKIP them: {orphan_corr}\n"
        "   Usually means the filename stem doesn't match the video, or the\n"
        "   bundle/video isn't uploaded. Corrections must be named\n"
        "   corrections_<videostem>.json (e.g. game1.mp4 -> corrections_game1.json).")

def block(title, items, mark):
    if not items: return
    print(f"\n{mark} {title}")
    for it in items:
        print("   " + it.replace("\n", "\n"))

block("MUST FIX before running (Cell 1 will stop):", problems, "❌")
block("Heads-up (notebook still runs, but read these):", warnings, "⚠️")

if not problems and not warnings:
    print("\n✅ Everything looks good.")

# ── stop here with a clear message if essentials are missing ─────────────────
import torch
if not torch.cuda.is_available():
    problems.append("No GPU. Fix: Runtime → Change runtime type → T4 GPU, then Run all.")
if not games and not problems:
    problems.append("No usable game (need at least one bundle WITH its video). "
                    "See the table above.")
if problems:
    raise SystemExit("\n⛔ Fix the ❌ items above, then Run all again.")

# ── essentials OK: install pipeline + report GPU ─────────────────────────────
print("\nGPU:", torch.cuda.get_device_name(0))
zipfile.ZipFile(f"{D}/vbpipe.zip").extractall("pipeline")
%pip -q install ultralytics lap
%pip -q install -e pipeline
print(f"\nready: {len(games)} game(s) for training {list(games)}; "
      f"corrections to score: {sorted(set(corrs) & set(games))}")


In [ ]:
# Cell 2 — mine physics-verified arcs from every game
# hi-res mining: detect at native 1080p (finds ~3x more of the fast ball, per
# the density test), but scale detections back to 1280x720 BEFORE physics
# verification, because ballcv's linking/parabola thresholds are tuned in
# 720 pixel space. So the mined arcs come out in 720 coords (Cell 3 keeps SC=1.5).
import sys, subprocess, numpy as np
sys.path.insert(0, "pipeline")
from vbpipe.track import _frames
from vbpipe import ballcv
from ultralytics import YOLO
model = YOLO(f"{D}/ball_model.pt")

MINE_HI_RES = True   # False = old 720/1088 mining (faster, lower recall)
_dec = (1920, 1080, 1920) if MINE_HI_RES else (1280, 720, 1088)
_sx, _sy = 1280 / _dec[0], 720 / _dec[1]   # scale detections back to 720 space

def mine(video, game, conf=0.08, fps=20.0):
    arcs, hard_neg = {}, []
    for ri, r in enumerate(game["rallies"]):
        if r.get("phase") == "warmup": arcs[str(ri)] = []; continue
        dets = []
        for t, fr in _frames(video, r["start"], r["end"]-r["start"], fps, _dec[0], _dec[1]):
            res = model.predict(fr, conf=conf, imgsz=_dec[2], verbose=False)[0]
            dets.append((round(t,3), [(float(b.xywh[0][0])*_sx, float(b.xywh[0][1])*_sy)
                                      for b in res.boxes]))
        pts = ballcv._select(ballcv._link(dets))     # runs in 720 space
        arcs[str(ri)] = [[p[0], p[1], p[2]] for p in pts]
        kept = set(round(p[0],1) for p in pts)
        hard_neg += [t for t,cs in dets if cs and round(t,1) not in kept][::7]
        print(f"  rally {ri}: {sum(len(c) for _,c in dets)} raw -> {len(pts)} verified")
    return arcs, hard_neg

print(f"mining at {'1080p/imgsz1920' if MINE_HI_RES else '720p/imgsz1088'}"
      f"  (arcs stored in 1280x720 coords)")
mined = {}
for stem, (video, g) in games.items():
    print(f"=== mining {stem} ===")
    mined[stem] = mine(video, g)
    n = sum(len(v) for v in mined[stem][0].values())
    dur = sum(r["end"]-r["start"] for r in g["rallies"] if r.get("phase") != "warmup")
    print(f"{stem}: {n} verified points ({n/max(dur,1):.1f}/s across rallies)")


In [ ]:
# Cell 3 — build combined dataset (positives + hard negatives per game)
import os, subprocess
OUT = "dataset"
os.makedirs(f"{OUT}/images/train", exist_ok=True)
os.makedirs(f"{OUT}/labels/train", exist_ok=True)
W, H, SC = 1920, 1080, 1.5   # mined points are 1280x720 -> label at 1080p

def grab(video, t, fn):
    subprocess.run(["ffmpeg","-v","error","-ss",str(t),"-i",video,
                    "-frames:v","1","-q:v","3",fn,"-y"])
    return os.path.exists(fn)

tot_pos = tot_neg = 0
for stem, (video, g) in games.items():
    arcs, hard_neg = mined[stem]
    items = sorted((p[0], p[1]*SC, p[2]*SC) for pts in arcs.values() for p in pts)
    sel, last = [], -1
    for t,x,y in items:                       # ~2 frames/sec cap
        if t - last >= 0.45: sel.append((t,x,y)); last = t
    n = 0
    for t,x,y in sel:
        fn = f"{OUT}/images/train/{stem}_{int(t*100):08d}.jpg"
        if not grab(video, t, fn): continue
        bw = 26 if y < H*0.6 else 40
        open(fn.replace("/images/","/labels/").replace(".jpg",".txt"),"w").write(
            f"0 {x/W:.5f} {y/H:.5f} {bw/W:.5f} {bw/H:.5f}\n")
        n += 1
    hn = 0
    for t in hard_neg[:max(1, int(n*0.4))]:   # model fired, physics said no
        fn = f"{OUT}/images/train/{stem}_hn{int(t*100):08d}.jpg"
        if not grab(video, t, fn): continue
        open(fn.replace("/images/","/labels/").replace(".jpg",".txt"),"w").close()
        hn += 1
    tot_pos += n; tot_neg += hn
    print(f"{stem}: {n} positives, {hn} hard negatives")
open(f"{OUT}/data.yaml","w").write(
    f"path: {os.path.abspath(OUT)}\ntrain: images/train\nval: images/train\nnames: {{0: ball}}\n")
print(f"dataset: {tot_pos} positives, {tot_neg} hard negatives")

In [ ]:
# Cell 4 — fine-tune from the current model, save as ball_model_v2.pt
# TRAIN AT 1080p (imgsz=1920): the dataset frames are already native 1080p, so
# training at 1920 teaches the ball at the SAME scale we run hi-res inference —
# fixing the precision collapse we saw when a 1088-trained model met 1080p input.
import glob, shutil
TRAIN_IMGSZ  = 1920   # match hi-res inference; 1088 = old low-res behavior
TRAIN_BATCH  = 4      # 1920 uses ~3x the memory of 1088 — lower to 2 if you OOM
TRAIN_EPOCHS = 40     # ~2-3x slower per epoch at 1920 than at 1088

m = YOLO(f"{D}/ball_model.pt")
m.train(data=f"{OUT}/data.yaml", epochs=TRAIN_EPOCHS, imgsz=TRAIN_IMGSZ,
        batch=TRAIN_BATCH, patience=15, mosaic=0.3, scale=0.2, degrees=0,
        fliplr=0.5, plots=False)
best = sorted(glob.glob("runs/detect/*/weights/best.pt"))[-1]
shutil.copy(best, f"{D}/ball_model_v2.pt")
print("saved", f"{D}/ball_model_v2.pt   (v1 untouched)   trained @ imgsz={TRAIN_IMGSZ}")


In [ ]:
# Cell 5 — score the candidate (v2, now trained @1080p) vs the current model (v1)
# The fair test is v2@1080: model trained at 1080p, run at 1080p. We also do a
# confidence sweep on the 1080p rows — the hi-res model over-fires at low conf,
# and the right threshold trades the excess back into precision. (We detect ONCE
# at a low floor and re-threshold in Python, so the sweep is basically free.)
from vbpipe.balltrain import detect_all
from vbpipe.plays import find_contacts
from collections import defaultdict

RUN_HI_RES = True                       # the 1080p passes are ~2-3x slower each
CONF_SWEEP = [0.15, 0.25, 0.35, 0.50]   # thresholds tried on the 1080p rows
BIG, MARGIN = 0.03, 0.01                # F1-gain decision thresholds (vs v1@720)

m1 = YOLO(f"{D}/ball_model.pt")      # current model (v1)
m2 = YOLO(f"{D}/ball_model_v2.pt")   # candidate, trained @1080p (Cell 4)

def contact_f1(ball, corr):
    by_idx = defaultdict(list)
    for cr in corr["rallies"]:
        if cr["phase"] == "game" and cr["idx"] is not None and cr["idx"] >= 0:
            by_idx[cr["idx"]] += [p["t"] for p in cr["plays"]]
    tp = fp = fn = 0
    for idx, truths in by_idx.items():
        pred = [c["t"] for c in find_contacts(ball[idx])] if idx < len(ball) else []
        used, mm = set(), 0
        for p in pred:
            best, bd = None, 0.5
            for j, t in enumerate(truths):
                if j not in used and abs(p-t) < bd: bd, best = abs(p-t), j
            if best is not None: used.add(best); mm += 1
        tp += mm; fp += len(pred)-mm; fn += len(truths)-mm
    P = tp/(tp+fp) if tp+fp else 0; R = tp/(tp+fn) if tp+fn else 0
    return P, R, (2*P*R/(P+R) if P+R else 0)

def dens(ball, dur): return sum(len(b) for b in ball) / max(dur, 1)
def filt(ball, c):   return [[p for p in b if len(p) < 4 or p[3] >= c] for b in ball]

for stem, corr in corrs.items():
    if stem not in games: print(f"skip {stem}: no video/bundle"); continue
    video, g = games[stem]
    dur = sum(r["end"]-r["start"] for r in g["rallies"] if r.get("phase") != "warmup")
    print(f"\n================  {stem}  ================")

    def row(label, ball):
        P, R, F = contact_f1(ball, corr)
        print(f"  {label:<9} density {dens(ball,dur):4.1f}/s   P={P:.0%} R={R:.0%} F1={F:.2f}")
        return F

    base = row("v1@720", g.get("ball", []))                  # baseline (from bundle)
    print("  (re-detecting v2 @720…)");  row("v2@720", detect_all(m2, video, g["rallies"]))

    cand_best = None
    if RUN_HI_RES:
        for tag, mdl in [("v1@1080", m1), ("v2@1080", m2)]:
            print(f"  (re-detecting {tag} @ conf floor 0.10…)")
            ball = detect_all(mdl, video, g["rallies"], hi_res=True, conf=0.10)
            print(f"    {tag} confidence sweep:")
            best = (-1, None)
            for c in CONF_SWEEP:
                fb = filt(ball, c)
                P, R, F = contact_f1(fb, corr)
                print(f"       conf {c:.2f}: density {dens(fb,dur):4.1f}/s  P={P:.0%} R={R:.0%} F1={F:.2f}")
                if F > best[0]: best = (F, c)
            print(f"    -> best {tag}: conf {best[1]:.2f}  F1={best[0]:.2f}")
            if tag == "v2@1080": cand_best = best

    # verdict: the candidate at its best 1080p operating point vs baseline
    if cand_best:
        d = cand_best[0] - base
        tag = ("✅ WIN — promote-worthy" if d >= BIG else
               "🟡 marginal — try more games / more epochs" if d >= MARGIN else
               "❌ no gain")
        print(f"  VERDICT {stem}: best v2@1080 F1={cand_best[0]:.2f} (conf {cand_best[1]:.2f})"
              f"  vs  v1@720 F1={base:.2f}   => {d:+.2f}  {tag}")

print("\n" + "="*56)
print("Decision (per game, contact F1 of best v2@1080 vs v1@720 baseline):")
print(f"  >= {BIG:+.2f}  ✅ the hi-res retrain works — run Cell 6 to promote v2,")
print(f"               and bake conf + hi_res into the pipeline (cli.py).")
print(f"  {MARGIN:+.2f}..{BIG:+.2f} 🟡 real but small — more reviewed games or epochs, re-run.")
print(f"  <  {MARGIN:+.2f}  ❌ resolution + retrain didn't pay off; pivot to tiling / data.")
print("Cell 6 promotes ball_model_v2.pt (the candidate). Only run it on a ✅/strong 🟡.")


In [ ]:
# Cell 6 — promote v2 (safe under Run all: does nothing until you flip the flag)
# Self-contained: works even if the runtime recycled since Cells 1-5 ran.
PROMOTE = False   # <-- set True and re-run this cell after checking Cell 5's numbers

if not PROMOTE:
    print("Not promoting (PROMOTE=False). Check Cell 5; if v2 wins, set PROMOTE=True and re-run this cell.")
else:
    import os, shutil
    if not os.path.exists("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
    D = "/content/drive/MyDrive/balltime"
    assert os.path.exists(f"{D}/ball_model_v2.pt"), "ball_model_v2.pt not in Drive - re-run Cells 1-4"
    shutil.copy(f"{D}/ball_model.pt", f"{D}/ball_model_v1_backup.pt")
    shutil.copy(f"{D}/ball_model_v2.pt", f"{D}/ball_model.pt")
    print("promoted (v1 backed up as ball_model_v1_backup.pt).")
    print("Now delete the bundles you want reprocessed from Drive/balltime/bundles")
    print("and Run all in process_game.ipynb (v8).")